# MATH 6243 — Modeling
## Predicting Personal Film Preferences from Letterboxd Ratings

**Structure:**
1. Setup & load data
2. Feature engineering
3. Label definitions (binary, 3-class, 4-class)
4. Cross-validation framework
5. Model 1 — Elastic Net Logistic Regression
6. Model 2 — Random Forest
7. Model 3 — From-Scratch Logistic Regression (binary)
8. Results comparison
9. Feature importance & coefficients

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, classification_report
)
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
SEED = 42

df = pd.read_csv('enriched_ratings.csv')

for col in ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime',
            'release_year', 'tmdb_popularity', 'budget', 'awards']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'{len(df)} films loaded')
df.head(3)

## 2. Feature Engineering

Features used:
- **Numeric:** `imdb_rating`, `rt_score`, `tmdb_vote_avg`, `runtime`, `release_year`, `awards`
- **Genre:** one-hot encoded from pipe-separated genre string
- **Language:** one-hot (top 5 languages + 'other')
- **Director:** target-encoded (mean rating per director) — computed inside CV folds to prevent leakage

Dropped: `budget`, `tmdb_popularity`, `tmdb_vote_count`, `imdb_votes` (noisy or redundant)

In [ ]:
# --- Genre one-hot ---
df['genres'] = df['genres'].fillna('')
genre_lists = df['genres'].apply(lambda x: [g.strip() for g in x.split('|') if g.strip()])

mlb = MultiLabelBinarizer()
genre_ohe = pd.DataFrame(
    mlb.fit_transform(genre_lists),
    columns=[f'genre_{g}' for g in mlb.classes_],
    index=df.index
)

# Drop genres appearing in fewer than 5 films (too sparse)
genre_ohe = genre_ohe.loc[:, genre_ohe.sum() >= 5]
print(f'Genre features kept: {genre_ohe.columns.tolist()}')

# --- Language one-hot (top 5 + other) ---
top_langs = df['original_language'].value_counts().head(5).index.tolist()
df['language_collapsed'] = df['original_language'].apply(
    lambda x: x if x in top_langs else 'other'
)
lang_ohe = pd.get_dummies(df['language_collapsed'], prefix='lang')
print(f'Language features: {lang_ohe.columns.tolist()}')

# --- Numeric features ---
numeric_cols = ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime', 'release_year', 'awards']

# --- Assemble base feature matrix (without director — added per fold) ---
X_base = pd.concat([df[numeric_cols], genre_ohe, lang_ohe], axis=1).copy()

print(f'\nBase feature matrix shape: {X_base.shape}')
print(f'Missing values per column (top):')
print(X_base.isnull().sum()[X_base.isnull().sum() > 0])

## 3. Label Definitions

In [ ]:
# Binary
y_binary = (df['rating'] >= 3.5).astype(int)

# 4-class: Disliked (<=2) / Mixed (<=3) / Liked (<=4) / Loved (>4)
def label_4(r):
    if r <= 2.0: return 0
    elif r <= 3.0: return 1
    elif r <= 4.0: return 2
    else: return 3

# 3-class: Disliked (<=2.5) / Mixed (<=3.5) / Liked (>3.5)
def label_3(r):
    if r <= 2.5: return 0
    elif r <= 3.5: return 1
    else: return 2

y_4class = df['rating'].apply(label_4)
y_3class = df['rating'].apply(label_3)

label_names = {
    'binary':  ['Not Liked', 'Liked'],
    '3class':  ['Disliked', 'Mixed', 'Liked'],
    '4class':  ['Disliked', 'Mixed', 'Liked', 'Loved'],
}

for name, y in [('Binary', y_binary), ('3-class', y_3class), ('4-class', y_4class)]:
    counts = y.value_counts().sort_index()
    pcts = (counts / len(y) * 100).round(1)
    print(f'{name}: {dict(zip(counts.index, zip(counts.values, pcts.values)))}')

## 4. Cross-validation Framework

Helper functions for:
- Director target encoding (within fold)
- Numeric imputation + scaling (within fold)
- Metric computation

In [ ]:
def director_target_encode(train_idx, val_idx, y_train_raw):
    """Compute mean raw rating per director on train fold, apply to both splits."""
    train_directors = df['director'].iloc[train_idx]
    val_directors   = df['director'].iloc[val_idx]
    global_mean     = y_train_raw.mean()

    dir_means = (
        pd.Series(y_train_raw.values, index=train_directors.values)
        .groupby(level=0).mean()
    )
    train_enc = train_directors.map(dir_means).fillna(global_mean).values
    val_enc   = val_directors.map(dir_means).fillna(global_mean).values
    return train_enc.reshape(-1,1), val_enc.reshape(-1,1)


def prepare_fold(train_idx, val_idx, y):
    """Full fold preparation: impute, scale, add director encoding."""
    X_tr = X_base.iloc[train_idx].copy()
    X_va = X_base.iloc[val_idx].copy()

    # Impute missing (median on train fold)
    imp = SimpleImputer(strategy='median')
    X_tr = pd.DataFrame(imp.fit_transform(X_tr), columns=X_base.columns)
    X_va = pd.DataFrame(imp.transform(X_va),     columns=X_base.columns)

    # Director target encoding
    y_raw_train = df['rating'].iloc[train_idx]
    dir_tr, dir_va = director_target_encode(train_idx, val_idx, y_raw_train)
    X_tr = np.hstack([X_tr.values, dir_tr])
    X_va = np.hstack([X_va.values, dir_va])

    # Scale
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)

    y_tr = y.iloc[train_idx].values
    y_va = y.iloc[val_idx].values
    return X_tr, X_va, y_tr, y_va


def compute_metrics(y_true, y_pred, y_prob, task):
    """Returns dict of metrics appropriate for the task."""
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)

    if task == 'binary':
        auc = roc_auc_score(y_true, y_prob[:, 1])
    else:
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
        except ValueError:
            auc = np.nan

    return {'auc': auc, 'accuracy': acc, 'f1_macro': f1}


def run_cv(model_fn, y, task, n_splits=5):
    """
    Run stratified k-fold CV.
    model_fn: callable that returns a fitted sklearn estimator given (X_tr, y_tr)
    Returns mean metrics dict.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_metrics = []

    for train_idx, val_idx in skf.split(X_base, y):
        X_tr, X_va, y_tr, y_va = prepare_fold(train_idx, val_idx, y)
        model = model_fn(X_tr, y_tr)
        y_pred = model.predict(X_va)
        y_prob = model.predict_proba(X_va)
        fold_metrics.append(compute_metrics(y_va, y_pred, y_prob, task))

    means = {k: np.mean([m[k] for m in fold_metrics]) for k in fold_metrics[0]}
    stds  = {k: np.std( [m[k] for m in fold_metrics]) for k in fold_metrics[0]}
    return means, stds, fold_metrics

print('CV framework ready.')

## 5. Model 1 — Elastic Net Logistic Regression

Tuning: `C` (inverse penalty strength) and `l1_ratio` (L1 vs L2 mix) via nested CV grid search.

In [ ]:
def tune_elasticnet(X_tr, y_tr, task, inner_splits=3):
    """Inner CV grid search for elastic net hyperparameters."""
    param_grid = {
        'C':          [0.01, 0.1, 0.5, 1.0, 5.0],
        'l1_ratio':   [0.1, 0.3, 0.5, 0.7, 0.9],
    }
    multi = 'ovr' if task != 'binary' else 'auto'
    inner_skf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=SEED)
    best_score, best_params = -np.inf, None

    for params in ParameterGrid(param_grid):
        scores = []
        for i_tr, i_va in inner_skf.split(X_tr, y_tr):
            m = LogisticRegression(
                penalty='elasticnet', solver='saga',
                C=params['C'], l1_ratio=params['l1_ratio'],
                multi_class=multi, max_iter=2000, random_state=SEED
            )
            m.fit(X_tr[i_tr], y_tr[i_tr])
            y_prob = m.predict_proba(X_tr[i_va])
            if task == 'binary':
                s = roc_auc_score(y_tr[i_va], y_prob[:, 1])
            else:
                try:
                    s = roc_auc_score(y_tr[i_va], y_prob, multi_class='ovr', average='macro')
                except:
                    s = 0
            scores.append(s)
        if np.mean(scores) > best_score:
            best_score = np.mean(scores)
            best_params = params

    return best_params


en_results = {}
en_best_params = {}

tasks = [
    ('binary',  y_binary,  'Binary (>=3.5)'),
    ('3class',  y_3class,  '3-class'),
    ('4class',  y_4class,  '4-class'),
]

for task_key, y, task_name in tasks:
    print(f'\nElastic Net — {task_name} ...')
    multi = 'ovr' if task_key != 'binary' else 'auto'

    def make_en(X_tr, y_tr, tk=task_key):
        params = tune_elasticnet(X_tr, y_tr, tk)
        m = LogisticRegression(
            penalty='elasticnet', solver='saga',
            C=params['C'], l1_ratio=params['l1_ratio'],
            multi_class='ovr' if tk != 'binary' else 'auto',
            max_iter=2000, random_state=SEED
        )
        m.fit(X_tr, y_tr)
        return m

    means, stds, _ = run_cv(make_en, y, task_key)
    en_results[task_name] = means
    print(f'  AUC={means["auc"]:.3f} (+/-{stds["auc"]:.3f})  '
          f'Acc={means["accuracy"]:.3f}  F1={means["f1_macro"]:.3f}')

## 6. Model 2 — Random Forest

Tuning: `n_estimators`, `max_depth`, `min_samples_leaf` via inner CV.

In [ ]:
def tune_rf(X_tr, y_tr, task, inner_splits=3):
    param_grid = {
        'n_estimators':    [100, 200],
        'max_depth':       [3, 5, None],
        'min_samples_leaf':[1, 3, 5],
    }
    inner_skf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=SEED)
    best_score, best_params = -np.inf, None

    for params in ParameterGrid(param_grid):
        scores = []
        for i_tr, i_va in inner_skf.split(X_tr, y_tr):
            m = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
            m.fit(X_tr[i_tr], y_tr[i_tr])
            y_prob = m.predict_proba(X_tr[i_va])
            if task == 'binary':
                s = roc_auc_score(y_tr[i_va], y_prob[:, 1])
            else:
                try:
                    s = roc_auc_score(y_tr[i_va], y_prob, multi_class='ovr', average='macro')
                except:
                    s = 0
            scores.append(s)
        if np.mean(scores) > best_score:
            best_score = np.mean(scores)
            best_params = params

    return best_params


rf_results = {}

for task_key, y, task_name in tasks:
    print(f'\nRandom Forest — {task_name} ...')

    def make_rf(X_tr, y_tr, tk=task_key):
        params = tune_rf(X_tr, y_tr, tk)
        m = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
        m.fit(X_tr, y_tr)
        return m

    means, stds, _ = run_cv(make_rf, y, task_key)
    rf_results[task_name] = means
    print(f'  AUC={means["auc"]:.3f} (+/-{stds["auc"]:.3f})  '
          f'Acc={means["accuracy"]:.3f}  F1={means["f1_macro"]:.3f}')

## 7. Model 3 — From-Scratch Logistic Regression (Binary)

Gradient descent implementation using NumPy only. Benchmarked against sklearn.

In [ ]:
class LogisticRegressionScratch:
    """
    Binary logistic regression via batch gradient descent.
    Loss: binary cross-entropy
    """
    def __init__(self, lr=0.01, max_iter=1000, tol=1e-5, random_state=42):
        self.lr           = lr
        self.max_iter     = max_iter
        self.tol          = tol
        self.random_state = random_state
        self.weights_     = None
        self.bias_        = None
        self.loss_curve_  = []

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def _bce_loss(y, y_hat):
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n, p = X.shape
        self.weights_ = rng.normal(0, 0.01, p)
        self.bias_    = 0.0
        self.loss_curve_ = []

        for i in range(self.max_iter):
            z     = X @ self.weights_ + self.bias_
            y_hat = self._sigmoid(z)
            err   = y_hat - y

            # Gradients
            dw = X.T @ err / n
            db = err.mean()

            self.weights_ -= self.lr * dw
            self.bias_    -= self.lr * db

            loss = self._bce_loss(y, y_hat)
            self.loss_curve_.append(loss)

            if i > 0 and abs(self.loss_curve_[-2] - loss) < self.tol:
                print(f'    Converged at iteration {i}')
                break

        return self

    def predict_proba(self, X):
        y_hat = self._sigmoid(X @ self.weights_ + self.bias_)
        return np.column_stack([1 - y_hat, y_hat])

    def predict(self, X):
        return (self._sigmoid(X @ self.weights_ + self.bias_) >= 0.5).astype(int)

    def classes_(self):
        return np.array([0, 1])

print('LogisticRegressionScratch defined.')

In [ ]:
# --- Run from-scratch vs sklearn on binary task ---

scratch_results_per_fold = []
sklearn_results_per_fold = []
loss_curves = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_base, y_binary)):
    X_tr, X_va, y_tr, y_va = prepare_fold(train_idx, val_idx, y_binary)

    # From-scratch
    scratch = LogisticRegressionScratch(lr=0.1, max_iter=2000, tol=1e-6)
    scratch.fit(X_tr, y_tr)
    loss_curves.append(scratch.loss_curve_)
    s_metrics = compute_metrics(y_va, scratch.predict(X_va), scratch.predict_proba(X_va), 'binary')
    scratch_results_per_fold.append(s_metrics)

    # sklearn (no tuning — same default C=1 for fair comparison)
    sk = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000, random_state=SEED)
    sk.fit(X_tr, y_tr)
    sk_metrics = compute_metrics(y_va, sk.predict(X_va), sk.predict_proba(X_va), 'binary')
    sklearn_results_per_fold.append(sk_metrics)

scratch_means = {k: np.mean([m[k] for m in scratch_results_per_fold]) for k in ['auc','accuracy','f1_macro']}
sklearn_means  = {k: np.mean([m[k] for m in sklearn_results_per_fold])  for k in ['auc','accuracy','f1_macro']}

print(f'From-scratch:  AUC={scratch_means["auc"]:.3f}  Acc={scratch_means["accuracy"]:.3f}  F1={scratch_means["f1_macro"]:.3f}')
print(f'sklearn:        AUC={sklearn_means["auc"]:.3f}  Acc={sklearn_means["accuracy"]:.3f}  F1={sklearn_means["f1_macro"]:.3f}')

In [ ]:
# Loss curves across folds
fig, ax = plt.subplots(figsize=(8, 4))
palette = sns.color_palette('muted', 5)
for i, curve in enumerate(loss_curves):
    ax.plot(curve, color=palette[i], alpha=0.8, linewidth=1.5, label=f'Fold {i+1}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('From-Scratch Logistic Regression — Loss Curves')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fig_loss_curves.png', bbox_inches='tight')
plt.show()

## 8. Results Comparison

In [ ]:
rows = []
for task_name, metrics in en_results.items():
    rows.append({'Model': 'Elastic Net', 'Task': task_name,
                 'AUC': round(metrics['auc'], 3),
                 'Accuracy': round(metrics['accuracy'], 3),
                 'F1 (macro)': round(metrics['f1_macro'], 3)})

for task_name, metrics in rf_results.items():
    rows.append({'Model': 'Random Forest', 'Task': task_name,
                 'AUC': round(metrics['auc'], 3),
                 'Accuracy': round(metrics['accuracy'], 3),
                 'F1 (macro)': round(metrics['f1_macro'], 3)})

rows.append({'Model': 'LR Scratch', 'Task': 'Binary (>=3.5)',
             'AUC': round(scratch_means['auc'], 3),
             'Accuracy': round(scratch_means['accuracy'], 3),
             'F1 (macro)': round(scratch_means['f1_macro'], 3)})

rows.append({'Model': 'LR sklearn (baseline)', 'Task': 'Binary (>=3.5)',
             'AUC': round(sklearn_means['auc'], 3),
             'Accuracy': round(sklearn_means['accuracy'], 3),
             'F1 (macro)': round(sklearn_means['f1_macro'], 3)})

results_df = pd.DataFrame(rows).sort_values(['Task', 'AUC'], ascending=[True, False])
results_df = results_df.reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
# Visual results table
fig, ax = plt.subplots(figsize=(10, 4))
metric_cols = ['AUC', 'Accuracy', 'F1 (macro)']

x = np.arange(len(results_df))
w = 0.25
palette = sns.color_palette('muted', 3)

for i, col in enumerate(metric_cols):
    ax.bar(x + i*w, results_df[col], width=w, label=col, color=palette[i], edgecolor='white')

ax.set_xticks(x + w)
labels = [f'{r["Model"]}\n({r["Task"]})' for _, r in results_df.iterrows()]
ax.set_xticklabels(labels, fontsize=7.5)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.axhline(0.70, color='crimson', linestyle='--', linewidth=1, alpha=0.6, label='AUC target (0.70)')
ax.legend(fontsize=8)
ax.set_title('Model Comparison — 5-Fold CV Metrics', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_comparison.png', bbox_inches='tight')
plt.show()

## 9. Feature Importance & Coefficients

Refit best models on full dataset for interpretation (inference focus).

In [ ]:
# Prepare full dataset (impute + scale + director encode on all data)
imp_full = SimpleImputer(strategy='median')
X_num = pd.DataFrame(imp_full.fit_transform(X_base), columns=X_base.columns)

global_mean = df['rating'].mean()
dir_means_full = df.groupby('director')['rating'].mean()
dir_enc = df['director'].map(dir_means_full).fillna(global_mean).values.reshape(-1, 1)

X_full = np.hstack([X_num.values, dir_enc])
feature_names = list(X_base.columns) + ['director_target_enc']

scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)

In [ ]:
# --- Elastic Net coefficients (binary) ---
en_full = LogisticRegression(
    penalty='elasticnet', solver='saga', C=0.5, l1_ratio=0.5,
    max_iter=2000, random_state=SEED
)
en_full.fit(X_full_scaled, y_binary)

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': en_full.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

# Top 20 by magnitude
top_coef = coef_df.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['seagreen' if c > 0 else 'tomato' for c in top_coef['coefficient']]
ax.barh(top_coef['feature'][::-1], top_coef['coefficient'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardized)')
ax.set_title('Elastic Net — Top 20 Coefficients (Binary Task)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_elasticnet_coefs.png', bbox_inches='tight')
plt.show()

print(coef_df.head(20).to_string(index=False))

In [ ]:
# --- Random Forest feature importances (binary) ---
rf_full = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=3,
    random_state=SEED, n_jobs=-1
)
rf_full.fit(X_full_scaled, y_binary)

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_full.feature_importances_
}).sort_values('importance', ascending=False)

top_imp = imp_df.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_imp['feature'][::-1], top_imp['importance'][::-1],
        color=sns.color_palette('muted')[0])
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('Random Forest — Top 20 Feature Importances (Binary Task)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rf_importances.png', bbox_inches='tight')
plt.show()

print(imp_df.head(20).to_string(index=False))